In [ ]:
# Enable autoreload of local Python modules
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import jax
import jax.numpy as jnp
import optax
import time
import matplotlib.pyplot as plt

from flax import linen as nn
from flax.training import train_state


In [ ]:
# Load text data
with open("./data/text8_train.txt", "r") as f:
    train_text = f.read()
with open("./data/text8_test.txt", "r") as f:
    test_text = f.read()

print(f"Length of training text: {len(train_text):_} characters")
print(f"Length of test text: {len(test_text):_} characters")

# Build vocabulary
char_set = list("abcdefghijklmnopqrstuvwxyz ")
char_to_int = {ch:i for i,ch in enumerate(char_set)}
int_to_char = {i:ch for ch,i in char_to_int.items()}

def encode(s):
    return np.array([char_to_int[c] for c in s], dtype=np.uint8)

train_text_int = encode(train_text)
test_text_int = encode(test_text)

# Display random snippets
T = 128
for _ in range(3):
    start = np.random.randint(0, len(train_text)-T)
    print(train_text[start:start+T], "\n")


In [ ]:
class LSTMModel(nn.Module):
    vocab_size: int
    d_model: int
    n_layers: int
    
    @nn.compact
    def __call__(self, x):
        # x: (B, T)
        emb = nn.Embed(self.vocab_size, self.d_model)(x)  # (B, T, d_model)
        
        # LSTM layers
        output = emb
        for _ in range(self.n_layers):
            output, _ = nn.LSTMCell()(output)
        
        # Final linear layer
        logits = nn.Dense(self.vocab_size)(output)  # (B, T, vocab_size)
        return logits


In [ ]:
def create_model(rng, vocab_size=27, d_model=256, n_layers=2):
    model = LSTMModel(vocab_size, d_model, n_layers)
    dummy = jnp.zeros((1, 32), dtype=jnp.int32)  # (B=1, T=32)
    params = model.init(rng, dummy)["params"]
    return model, params

key = jax.random.PRNGKey(0)
vocab_size = len(char_set)

model, params = create_model(key, vocab_size=vocab_size, d_model=256, n_layers=2)

# Count parameters
def count_params(params):
    return sum(x.size for x in jax.tree_util.tree_leaves(params))

print(f"Number of parameters: {count_params(params):_}")


In [ ]:
@jax.jit
def loss_and_metrics(logits, targets):
    vocab = logits.shape[-1]
    flat_logits = logits.reshape(-1, vocab)
    flat_targets = targets.reshape(-1)
    
    per_pos = optax.softmax_cross_entropy_with_integer_labels(flat_logits, flat_targets)
    loss = per_pos.mean()
    
    preds = jnp.argmax(logits, axis=-1)  # (B, T)
    is_match = preds == targets
    acc_all = jnp.mean(is_match.astype(jnp.float32))
    acc_last = jnp.mean(is_match.astype(jnp.float32)[:, -1])
    
    return loss, {"loss": loss, "acc": acc_all, "acc_last": acc_last}


In [ ]:
def train_step(params, opt_state, x, y, tx):
    def loss_fn(params):
        logits = model.apply({"params": params}, x)
        loss, metrics = loss_and_metrics(logits, y)
        return loss, metrics
    
    (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    updates, new_opt_state = tx.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, metrics

train_step = jax.jit(train_step, static_argnames=("tx",))


In [ ]:
def get_batch(text_int, B, T):
    ix = np.random.randint(0, len(text_int) - T, size=B)
    x = np.stack([text_int[i:i+T] for i in ix])
    y = np.stack([text_int[i+1:i+T+1] for i in ix])
    return jnp.array(x, dtype=jnp.int32), jnp.array(y, dtype=jnp.int32)


In [ ]:
learning_rate = 0.001
tx = optax.adam(learning_rate=learning_rate)
opt_state = tx.init(params)

print(f"Initialized Adam optimizer with lr={learning_rate}")


In [ ]:
niter = 50_000
B, T = 128, 32
loss_history = []
time_history = []
loss_test_history = []
time_test_history = []

start_time = time.time()

for it in range(niter):
    input, target = get_batch(train_text_int, B, T)
    params, opt_state, metrics = train_step(params, opt_state, input, target, tx)
    
    loss = metrics['loss']
    acc = metrics['acc']
    acc_last = metrics['acc_last']
    
    loss_history.append(loss)
    time_history.append(time.time() - start_time)

    if it % (niter // 50) == 0 or it == niter - 1:
        B_test, T_test = 512, 32
        test_input, test_target = get_batch(test_text_int, B_test, T_test)
        test_logits = model.apply({"params": params}, test_input)
        test_loss, test_metrics = loss_and_metrics(test_logits, test_target)
        
        print(f"it {it:_} | loss {loss:.4f} | test {test_loss:.4f} | "
              f"acc {acc:.3f} | test_acc {test_metrics['acc']:.3f}")


In [ ]:
plt.plot(time_history, loss_history, label='train', lw=1)
plt.plot(time_test_history, loss_test_history, label='test', lw=2)
plt.xlabel("Time (sec)")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.title("Training Loss (LSTM)")
plt.show()


In [ ]:
def generate_lstm(model, params, prompt, gen_len=200, temperature=0.7):
    ids = encode(prompt.lower())
    x = jnp.array(ids, dtype=jnp.int32).reshape(1, -1)

    for _ in range(gen_len):
        logits = model.apply({"params": params}, x)
        last = logits[:, -1, :] / temperature
        probs = jax.nn.softmax(last, axis=-1)
        
        next_id = jax.random.categorical(jax.random.PRNGKey(time.time_ns()), last)
        x = jnp.concatenate([x, next_id.reshape(1, 1)], axis=1)
    
    return ''.join(int_to_char[int(i)] for i in list(x[0]))

prompt = "hello my fri"
generated = generate_lstm(model, params, prompt, gen_len=500, temperature=0.7)
print(prompt + generated)
